<a href="https://colab.research.google.com/github/eric56427/er1c/blob/main/Project_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Intializaiton Phase 1

SET parameters:
    N = 60                           
    p = 0.05                         
    L_range = (5, 25)                
    T = 100                          
    simulation_ticks = 36000         
    f = 0.6                          
    k = 3                           

Create Graph representing the city network

    FUNCTION create_connected_graph(N, p, L_range):
    Do:
        graph = GENERATE random graph with N nodes and edge prob p
    While graph is not connected

    FOR each edge in graph:
        ASSIGN random length from uniform distribution in L_range

    RETURN graph

Initialize traffic counting dict

traffic_counts = {}
    FOR each edge in graph:
    traffic_counts[edge] = 0

Data Collection Phase

FOR tick FROM 1 TO simulation_ticks:

    FOR trip FROM 1 TO T:
        Generate random distinct start and end locations
    DO:
            start = RANDOM node from graph
            end = RANDOM node from graph
    WHILE start == end
        
         Find shortest path using A* algorithm
            path = A_STAR(graph, start, end)
        
        // Increment traffic counts for each edge in path
    FOR each consecutive pair (u, v) in path:
            edge = (min(u,v), max(u,v))  // Normalize edge representation
            traffic_counts[edge] += 1

Benefit Calculation for unconnected roads

    unconnected_pairs = GET all node pairs without direct connection
    benefits_matrix = {}

    FOR each unconnected pair (X, Y):
      spd_XY = SHORTEST_PATH_LENGTH(graph, X, Y)  // Using A*
      d_XY = spd_XY * f                           // New road length
    
     Calculate direct benefit
      direct_benefit = (spd_XY - d_XY) * (traffic_counts.get((X,Y),0) + traffic_counts.get((Y,X),0))
    
     Calculate neighbor benefits for X's neighbors
      neighbor_benefit_X = 0
    FOR each neighbor n2 of X:
        spd_Yn2 = SHORTEST_PATH_LENGTH(graph, Y, n2)
        potential_savings = spd_Yn2 - d_XY - graph.edges[(X, n2)]['length']
        IF potential_savings > 0:
            neighbor_benefit_X += potential_savings * (traffic_counts.get((Y,n2),0) + traffic_counts.get((n2,Y),0))
    
     Calculate neighbor benefits for Y's neighbors
    neighbor_benefit_Y = 0
    FOR each neighbor n1 of Y:
        spd_Xn1 = SHORTEST_PATH_LENGTH(graph, X, n1)
        potential_savings = spd_Xn1 - d_XY - graph.edges[(Y, n1)]['length']
        IF potential_savings > 0:
            neighbor_benefit_Y += potential_savings * (traffic_counts.get((X,n1),0) + traffic_counts.get((n1,X),0))
    
     Total benefit
    total_benefit = direct_benefit + neighbor_benefit_X + neighbor_benefit_Y
    benefits_matrix[(X,Y)] = total_benefit

Algorithim for road reccomendation

    recommended_roads = []
    built_roads = []

FOR i FROM 1 TO k:
    IF benefits_matrix is empty:
        BREAK
        
     Find road with the maximum benefit
    best_road = FIND entry in benefits_matrix with the maximum benefit value
    recommended_roads.APPEND(best_road)
    built_roads.APPEND(best_road)
    
     Update graph with the new road
    ADD edge best_road to graph with length = spd(best_road) * f
    
     Update benefits matrix for affected roads
    FOR each unconnected pair (X,Y) in benefits_matrix:
        IF road (X,Y) would be affected by new road best_road:
            RECALCULATE benefit for (X,Y) using updated graph
            UPDATE benefits_matrix[(X,Y)] with new value
    
    REMOVE best_road from benefits_matrix

Outputs in theory

PRINT "Recommended roads to build along with their benefits:"
FOR each road in recommended_roads:
    PRINT "Road:", road, "| Benefit:", benefits_matrix[road]